In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

In [3]:
PROJECT_ROOT = Path().resolve().parents[1]
PROJECT_ROOT

WindowsPath('C:/Users/User/workspace/portfolio/stroke_prediction')

In [4]:
DATA_DIR = PROJECT_ROOT / os.getenv("DATA_DIR")
DATA_DIR

WindowsPath('C:/Users/User/workspace/portfolio/stroke_prediction/data')

In [5]:
print(Path().resolve())

C:\Users\User\workspace\portfolio\stroke_prediction\experiments\notebooks


In [6]:
os.getcwd()

'c:\\Users\\User\\workspace\\portfolio\\stroke_prediction\\experiments\\notebooks'

In [7]:
df_stroke = pd.read_csv(DATA_DIR/"raw"/"healthcare-dataset-stroke-data.csv")

In [8]:
df_stroke.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


In [9]:
df_stroke.head(10)

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
5,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1
6,53882,Male,74.0,1,1,Yes,Private,Rural,70.09,27.4,never smoked,1
7,10434,Female,69.0,0,0,No,Private,Urban,94.39,22.8,never smoked,1
8,27419,Female,59.0,0,0,Yes,Private,Rural,76.15,NaN,Unknown,1
9,60491,Female,78.0,0,0,Yes,Private,Urban,58.57,24.2,Unknown,1


In [10]:
df_stroke.isnull().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [11]:
##This shows class imbalance
df_stroke["stroke"].value_counts(normalize=True)

stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64

In [12]:
#num_cols = ["age","avg_glucose_level","hypertension","heart_disease","bmi"]
num_cols = ["age","avg_glucose_level","bmi"]
cat_cols = ["gender","ever_married","smoking_status","work_type","Residence_type"]

In [14]:
for col in cat_cols:
    ct = pd.crosstab(df_stroke[col], df_stroke["stroke"], normalize="index")
    print(ct.round(2))

stroke     0     1
gender            
Female  0.95  0.05
Male    0.95  0.05
Other   1.00  0.00
stroke           0     1
ever_married            
No            0.98  0.02
Yes           0.93  0.07
stroke              0     1
smoking_status             
Unknown          0.97  0.03
formerly smoked  0.92  0.08
never smoked     0.95  0.05
smokes           0.95  0.05
stroke            0     1
work_type                
Govt_job       0.95  0.05
Never_worked   1.00  0.00
Private        0.95  0.05
Self-employed  0.92  0.08
children       1.00  0.00
stroke             0     1
Residence_type            
Rural           0.95  0.05
Urban           0.95  0.05


#Age and glucose level have strong relation with stroke.  BMI is not that impactful.

In [16]:
df_stroke.groupby("stroke")[num_cols].median()

,age,avg_glucose_level,bmi
stroke,,,
0,43.0,91.47,28.0
1,71.0,105.22,29.7


In [17]:

df_stroke.groupby("stroke")[num_cols].mean()

,age,avg_glucose_level,bmi
stroke,,,
0,41.971545,104.795513,28.823064
1,67.728193,132.544739,30.471292


The mean and median analysis shows average glucose level has outliers.

In [18]:
#Make a copy of a dataset, fill in missing bmi with median
df = df_stroke.copy()
df = df.drop(columns=["id"])
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

In [19]:
#Label encoding for binary variables
df["gender"] = df["gender"].map({"Male":0, "Other": 0, "Female":1})
df["ever_married"] = df["ever_married"].map({"No":0, "Yes":1})
df["Residence_type"] = df["Residence_type"].map({"Rural":0, "Urban":1})

In [20]:
#One hot encoding for categorical variables
df = pd.get_dummies(df,columns=["work_type","smoking_status"], drop_first=True)

In [23]:
#Define X (predictor) and y (Target) variables
X = df.drop(columns=["stroke"])
y = df["stroke"]

In [24]:
#Train/Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [25]:
#just a quick validation for the class distribution and shape,to confirm the train and test data split
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(4088, 15) (1022, 15)
stroke
0    0.951321
1    0.048679
Name: proportion, dtype: float64
stroke
0    0.951076
1    0.048924
Name: proportion, dtype: float64


Baseline model predict 95% accuracy, because it ignores the minority class.  It says I am prdicting all the NO stroke cases correctly, so I am 95% accurate.  But this is misleading.  
Missing 49 of the 50 actual stroke cases is a very serious problem with this model.

In [ ]:
#Baseline model Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)

print(cm_df)

print("\nClassification Report: \n", classification_report(y_test, y_pred))

Accuracy: 0.952054794520548
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  972                 0
Actual Stroke (FN/TP)                      49                 1

Classification Report: 
               precision    recall  f1-score   support

           0       0.95      1.00      0.98       972
           1       1.00      0.02      0.04        50

    accuracy                           0.95      1022
   macro avg       0.98      0.51      0.51      1022
weighted avg       0.95      0.95      0.93      1022



Handled the class imbalance with class_weight=balanced option.  Improvement in actual stroke prdiction cases, recall is 80%.  But accuracy went down to 74%.  False alarm 249, but it is better than missing actual cases.

In [27]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)

print(cm_df)

print("\nClassification Report: \n", classification_report(y_test, y_pred))

Accuracy: 0.7465753424657534
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  723               249
Actual Stroke (FN/TP)                      10                40

Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022



In [42]:
y_pred = model.predict(X_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(1022,))

In [29]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(1022,))

Introduced threshold tuning.  Recall went up, but more false alarms.  Find a best trade-off.  Seems, 0.5 is the sweet spot.

In [38]:
y_prob = model.predict_proba(X_test)
y_prob

array([[0.5374381 , 0.4625619 ],
       [0.81980628, 0.18019372],
       [0.91508669, 0.08491331],
       ...,
       [0.73314929, 0.26685071],
       [0.96034179, 0.03965821],
       [0.66544719, 0.33455281]], shape=(1022, 2))

In [39]:
y_prob = model.predict_proba(X_test)[:, 1]
y_prob

array([0.4625619 , 0.18019372, 0.08491331, ..., 0.26685071, 0.03965821,
       0.33455281], shape=(1022,))

In [40]:
y_pred_03 = (y_prob >= 0.3).astype(int)
y_pred_03

array([1, 0, 0, ..., 0, 0, 1], shape=(1022,))

In [41]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred_03 = (y_prob >= 0.3).astype(int)
print("\nThreshold = 0.3")

cm = confusion_matrix(y_test, y_pred_03)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)
print(cm_df)

#print(classification_report(y_test, y_pred_03))


Threshold = 0.3
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  581               391
Actual Stroke (FN/TP)                       8                42


In [31]:
y_prob

array([0.4625619 , 0.18019372, 0.08491331, ..., 0.26685071, 0.03965821,
       0.33455281], shape=(1022,))

In [32]:
y_pred_03

array([1, 0, 0, ..., 0, 0, 1], shape=(1022,))

In [33]:
y_pred_04 = (y_prob >= 0.4).astype(int)
print("\nThreshold = 0.4")

cm = confusion_matrix(y_test, y_pred_04)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)
print(cm_df)

print(classification_report(y_test, y_pred_04))


Threshold = 0.4
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  661               311
Actual Stroke (FN/TP)                       8                42
              precision    recall  f1-score   support

           0       0.99      0.68      0.81       972
           1       0.12      0.84      0.21        50

    accuracy                           0.69      1022
   macro avg       0.55      0.76      0.51      1022
weighted avg       0.95      0.69      0.78      1022



In [34]:
y_pred_05 = (y_prob >= 0.5).astype(int)
print("\nThreshold = 0.5")

cm = confusion_matrix(y_test, y_pred_05)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)
print(cm_df)

print(classification_report(y_test, y_pred_05))


Threshold = 0.5
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  723               249
Actual Stroke (FN/TP)                      10                40
              precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022



Try a better model - Random Forest.  Very interesting - RF completely ignored the minority class even with the class_weight=balanced.  Not a good choice.  Complex model is not always better.

In [64]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy", accuracy_score(y_test, y_pred_rf))

cm = confusion_matrix(y_test, y_pred_rf)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)
print(cm_df)

print(classification_report(y_test, y_pred_rf))

Accuracy 0.9500978473581213
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  971                 1
Actual Stroke (FN/TP)                      50                 0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022



In [35]:
df.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'Residence_type', 'avg_glucose_level', 'bmi', 'stroke',
       'work_type_Never_worked', 'work_type_Private',
       'work_type_Self-employed', 'work_type_children',
       'smoking_status_formerly smoked', 'smoking_status_never smoked',
       'smoking_status_smokes'],
      dtype='str')